In [1]:
from pathlib import Path
import csv
from datetime import datetime
import shutil

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType, StructField, StructType

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

input_path = PROJECT_ROOT / "data" / "raw" / "reddit" / "pushshift_1"
output_path = PROJECT_ROOT / "data" / "processed" / "crosspost_result"

spark = (
    SparkSession.builder
    .appName("reddit-crosspost-weight-graph-local")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)

start_time = datetime.now()

if not input_path.exists():
    raise FileNotFoundError(f"Input folder not found: {input_path}")

file_paths = sorted(input_path.glob("*_submissions_cleaned.parquet"))
if not file_paths:
    raise FileNotFoundError(f"No submission parquet files found in: {input_path}")

if output_path.exists():
    shutil.rmtree(output_path)

file_uris = [path.as_uri() for path in file_paths]
required_schema = StructType([
    StructField("name", StringType(), True),
    StructField("subreddit", StringType(), True),
    StructField("crosspost_parent", StringType(), True),
])

df_raw = spark.read.schema(required_schema).parquet(*file_uris)
df_submissions = df_raw.filter(F.col("name").startswith("t3_"))

# Lookup table: original post ID and source subreddit.
df_lookup = df_submissions.select(
    F.col("name").alias("original_id"),
    F.col("subreddit").alias("source_subreddit"),
).na.drop(subset=["original_id"])

# Edge table: posts whose crosspost_parent points to an original post.
df_crossposts = df_submissions.select(
    F.col("crosspost_parent"),
    F.col("subreddit").alias("target_subreddit"),
).filter(
    F.col("crosspost_parent").isNotNull()
    & (F.col("crosspost_parent") != "None")
    & (F.col("crosspost_parent") != "")
)

df_joined = df_lookup.join(
    df_crossposts,
    df_lookup["original_id"] == df_crossposts["crosspost_parent"],
    how="inner",
)

# Edge weight = number of times a source subreddit was crossposted to a target subreddit.
df_network = (
    df_joined.groupBy("source_subreddit", "target_subreddit")
    .count()
    .withColumnRenamed("count", "weight")
).persist()

df_network.orderBy(F.col("weight").desc()).show(20, truncate=False)

output_path.mkdir(parents=True, exist_ok=True)
with (output_path / "part-00000.csv").open("w", newline="", encoding="utf-8") as output_file:
    writer = csv.writer(output_file)
    writer.writerow(["source_subreddit", "target_subreddit", "weight"])
    writer.writerows(
        (row.source_subreddit, row.target_subreddit, row.weight)
        for row in df_network.toLocalIterator()
    )

df_network.unpersist()

end_time = datetime.now()
print(start_time)
print(end_time)
print(f"Execution time: {end_time - start_time}")
print(f"Output written to: {output_path}")

+--------------------+--------------------+------+
|source_subreddit    |target_subreddit    |weight|
+--------------------+--------------------+------+
|BABYDOGEARMY        |BabyDogeCoin        |2611  |
|China_Debate        |ChunghwaMinkuo      |2244  |
|ChunghwaMinkuo      |China_Debate        |1719  |
|CelebrityBelly      |Celebhub            |1173  |
|AbsurdMovies        |80s                 |930   |
|Celebhub            |1998TeenMovie       |832   |
|90dayfiance_FB_memes|90DayFianceSnark    |815   |
|Celebhub            |1110AsleepShower    |643   |
|ConservativeMemes   |Conservatives_R_Us  |638   |
|AvatarMemebending   |AvatarMemes         |622   |
|CelebEvents         |1110AsleepShower    |510   |
|Celebhub            |CelebrityBelly      |502   |
|AltarOfVenus        |1110AsleepShower    |501   |
|CPTSDmemes          |BPDmemes            |476   |
|ConservativeMemes   |AskThe_Donald       |455   |
|ConservativeMemes   |BidenBuzz           |404   |
|BPDmemes            |CPTSDmeme